<a href="https://colab.research.google.com/github/carn51/Assignment-1/blob/main/2026_DSLab2_a3(1)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DS Lab2 - Assignment 3: Making a Discovery in the AirBnB Dataset**

This is the final assessment for DSLab2, giving you chance to bring together the full skillset you have acquired over the course of the module. Note that this assignment makes up the final 40% of the grade for the module. \\

Please make sure you use the software versions specified in this notebook, rather than the latest or other versions.

**Task**

**Visit the Inside AirBnB website and choose at least 3 locations in Australia that you would be interested in comparing.** \\

(Note - if you have problems in Chrome try downloading in Safari or some other browser) \\

You may employ any of the tools we have encountered so far in the module. Choose or switch interchangably between Pandas, PySpark, Dask, and PyTorch, as well as all the usual Python packages (e.g. numpy, matplotlib), and string manipulation tools if required. \\

A greater degree of independence is expected in choosing the correct tools for a particular task in this assignment, and your analysis should be thorough and well documented. Marks will be assigned for the quality of your code, including clarity, commenting and appropriate visualisations e.g. histograms, scatterplots etc. \\

**To Submit:**

1.   **This notebook containing your code and visualizations**
      - Please name the file "YOUR_NAME_DSLab226_a3.ipynb" (i.e. update "YOUR NAME" to be your name...)
      - The file *must* be in .ipynb format - markers will check that your code runs successfully.

2.  **All data files that are needed to run your code**
      - e.g. If you work on the detailed reviews from Sydney, **you must submit the csv file** of Sydney's detailed reviews with your code and report so we can run the code.

3.   **1-page report summarising your work and findings**
      - Please name the file "YOUR_NAME_DSLab226_a3_report.pdf" (i.e. update "YOUR NAME" to be your name...)
      - The file *must* be submitted in PDF format - LaTEX is recommended to generate your report, but you will not be penalised for a well-presented Word document exported as a PDF.
      - Please aim to provide your full report within 2 sides of A4 (marks will be lost for reports less than 1 side of A4 or for reports greater than 2 sides of A4.)



# Setup

In [2]:
## Generic setup that we have used every week - this time they need to set up themselves
## Mount your Google Drive so we can access the data
from google.colab import drive
drive.mount('/content/drive',  force_remount=True)

Mounted at /content/drive


In [3]:
## Import useful python modules
import pandas as pd ## to read in and explore data
import numpy as np ## Useful for calculations
import matplotlib.pyplot as plt ## Useful for plotting
import matplotlib.gridspec as gridspec ## Useful for plotting

# Q1 - Classification Problem
**25 marks for developing appropriate code for the chosen problem**

**25 marks for a report documenting your choice of problem, motivation, rationale for the developed solution, results and conclusions**

Design and implement a classification problem that can be applied to each of your locations. You may focus on any problem that interests you, but make sure to include the following elements:


*   Clean the input dataframes by doing the following:

> Remove NaNs in your columns of interest \\
> Update the Dtype of any columns if necessary \\

*   Choose a suitable classification algorithm
*   Visualise the output of your classifications using at least one appropriate plot
*   Apply your technique to at least two different locations
*   Describe the problem, the approach you have chosen, and the results in your accompanying report
*   Remember to comment your code to help the markers (and yourself!). Think about using functions wherever appropriate to make it easier to apply your code to multiple locations for example. \\

\\

**If you require inspiration, perhaps consider one of the following problems**

(no marks gained or lost for using a suggested problem vs an original problem):

1.   By treating "`review_score_rating`" as a discrete variable (i.e. a series of distinct bins) construct a model to predict which of your chosen categories a given listing will fall into, e.g. excellent, good, intermediate, poor, rubbish or similar. \\

2.   Use sentiment analysis to classify reviews (i.e. "`comments`" column) into positive, neutral, or negative sentiment categories. Compare the overall satisfaction level with AirBnB listings across different cities (probably best to use at least 3 locations).  \\


Feel free to develop either of these ideas or one of your own, and see where the analysis takes you. You may find it helpful to start by mapping out in comments the broad-brush steps of your analysis, then think about how to accomplish each stage. \\

\\



# Load Data

In [126]:
## Load first chosen dataset into a Pandas (or PySpark/Dask) DataFrame

data_syd = pd.read_csv('/content/listings_syd.csv.gz')
df_syd = pd.DataFrame(data_syd)

data_bri = pd.read_csv('/content/listings_bri.csv.gz')
df_bri = pd.DataFrame(data_bri)

#data_mel = pd.read_csv('/content/listings_mel.csv.gz')
#df_mel = pd.DataFrame(data_mel)

data_que = pd.read_csv('/content/listings_que.csv.gz')
df_que = pd.DataFrame(data_que)

In [ ]:
## A couple of lines to orientate oneself wrt the data, columns and dtypes.



# Clean Data

In [130]:
## Do any appropriate cleaning

#adding a city column
df_syd['city'] = 'Sydney'
df_que['city'] = 'Queensland'
df_mel['city'] = 'Melbourne'

df0 = pd.concat([df_syd, df_que, df_mel], axis=0)
#df.head(3)

#select relevant columns
cdf = df0[['reviews_per_month', 'city', 'instant_bookable', 'availability_365', 'host_since', 'first_review', 'last_review']]
cdf = cdf.dropna().reset_index(drop=True)
cdf.head(3)

,reviews_per_month,city,instant_bookable,availability_365,host_since,first_review,last_review
0,1.01,Sydney,f,364,2009-09-23,2009-12-05,2020-03-13
1,3.83,Sydney,t,295,2009-12-03,2012-02-23,2025-09-01
2,0.47,Sydney,f,0,2010-04-22,2010-10-20,2025-08-31


Feature Engineering

In [131]:
import datetime

#current date for comparison
current = datetime.datetime.now()

#conversions to correct type
cdf['first_review'] = pd.to_datetime(cdf['first_review'])
cdf['last_review'] = pd.to_datetime(cdf['last_review'])
cdf['host_since'] = pd.to_datetime(cdf['host_since'])

cdf['instant_bookable'] = cdf['instant_bookable'].astype('category')
cdf['city'] = cdf['city'].astype('category')

#finding time in business in years
cdf['open_days'] = (cdf['last_review'] - cdf['first_review']) / np.timedelta64(1, 'D')
cdf['open_years'] = cdf['open_days'] /365

#experience of hosts
cdf['host_exp'] = ((current - cdf['host_since']) / pd.Timedelta(days= 365.25)).astype('float')

#availability to %
cdf['avail_pc'] = cdf['availability_365'] / 365

#drop intermediate columns
cdf = cdf.drop(columns = ['availability_365',	'host_since',	'first_review' ,'last_review' ,'open_days'])

In [132]:
#to numeric
cdf['avail_pc'] = pd.to_numeric(cdf['avail_pc'])
cdf['reviews_per_month'] = pd.to_numeric(cdf['reviews_per_month'])
cdf['open_years'] = pd.to_numeric(cdf['open_years'])
cdf['host_exp'] = pd.to_numeric(cdf['host_exp'])

In [133]:
cdf.head()

cdf['city'].unique()

['Sydney', 'Quuensland', 'Melbourne']
Categories (3, object): ['Melbourne', 'Quuensland', 'Sydney']

Encode

In [134]:
from sklearn.preprocessing import OneHotEncoder
#categorical: OneHot

#reassign as categorical
cdf['city'] = cdf['city'].astype('category')
cdf['instant_bookable'] = cdf['instant_bookable'].astype('category')

catcols = cdf.select_dtypes(include='category').columns.tolist()
oh = OneHotEncoder(sparse_output=False)
print(catcols)

#fit to relevant columns
oh_done = oh.fit_transform(cdf[catcols])
cols = oh.get_feature_names_out(catcols)

#onehot df to concat back
oh_df = pd.DataFrame(oh_done, columns=cols)

from sklearn.preprocessing import LabelEncoder
#numerical target: Label

le = LabelEncoder()

#fit numerical columns
le_done = le.fit_transform(cdf['reviews_per_month'])

#df to concat back
le_df = pd.DataFrame(le_done)

df = pd.concat([cdf, oh_df], axis=1).drop(columns = ['city', 'instant_bookable'])
df.head()

['city', 'instant_bookable']


,reviews_per_month,open_years,host_exp,avail_pc,city_Melbourne,city_Quuensland,city_Sydney,instant_bookable_f,instant_bookable_t
0,1.01,10.276712,16.575926,0.997260,0.0,0.0,1.0,1.0,0.0
1,3.83,13.531507,16.381538,0.808219,0.0,0.0,1.0,0.0,1.0
2,0.47,14.873973,15.998239,0.000000,0.0,0.0,1.0,1.0,0.0
3,2.50,14.682192,15.456145,0.378082,0.0,0.0,1.0,1.0,0.0
4,0.69,13.975342,15.297350,0.726027,0.0,0.0,1.0,1.0,0.0


Split

In [ ]:
#feature definition


# Classification

In [ ]:
## Write your code here

In [ ]:
## Add as many boxes as you like|

#Q2 - Regression Problem

**25 marks for developing appropriate code for the chosen problem**

**25 marks for a report documenting your choice of problem, motivation, rationale for the developed solution, results and conclusions** \\

Design and implement a regression problem that can be applied to each of your locations (and potentially to a merged set of locations). You may focus on any problem that interests you, but make sure to include the following elements:


*   Clean the input dataframes by doing the following:

> Remove NaNs in your columns of interest \\
> Update the Dtype of any columns if necessary \\

*   Choose a suitable regression algorithm
*   Decide whether to include categorical data or not, and deal with it appropriately
*   Visualise the output of your algorithm using at least one appropriate plot per location
*   Apply your technique to at least two different locations
*   Evaluate the results
*   Describe the problem, the approach you have chosen, and the results in your accompanying report
*   Remember to comment your code to help the markers (and yourself!). Think about using functions wherever appropriate to make it easier to apply your code to multiple locations for example. \\

\\

**If you require inspiration, perhaps consider the following**

(no marks gained or lost for using a suggested problem vs an original problem):

* Either of the problems suggested in section one could be treated as a
regression problem. For instance, the average `review_score_rating` or `average sentiment` are esssentially continuous variables. If you used your own problem instead you might also consider whether it is possible to adapt it into a regression problem. \\


You may find it helpful to start by mapping out in comments the broad-brush steps of your analysis, then think about how to accomplish each stage. For certain tasks it may be appropriate to utilise the GPUs available through Google colab. To avoid being thrown off, perhaps develop your code using the CPU first, and speed things up when you have a working setup to explore your idea and the results. \\

\\



# Load Data

In [ ]:
## Load first chosen dataset into a Pandas (or PySpark/Dask) DataFrame



# Clean Data

In [ ]:
## Do any appropriate cleaning here


# Regression Problem

In [ ]:
## Write your code here


In [ ]:
## Add as many boxes as you like